<a href="https://colab.research.google.com/github/Adeel213/Deep-learning/blob/main/Deep_Learning(CNN_Cats_vs_Dogs).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
import tensorflow_datasets as tfds
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
!pip install --upgrade --force-reinstall tensorflow_datasets protobuf

  Using cached tensorflow_datasets-4.9.10-py3-none-any.whl.metadata (11 kB)
  Using cached protobuf-7.36.1-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
  Using cached absl_py-2.5.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached array_record-0.8.3-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.1 kB)
  Using cached dm_tree-0.1.10-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.6 kB)
  Using cached etils-1.14.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached immutabledict-4.3.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached numpy-2.5.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached promise-2.3-py3-none-any.whl
  Using cached psutil-7.2.2-cp36-abi3-manylinux2010_x86_64.manylinux_2_12_x86_64.manylinux_2_28_x86_64.whl.metadata (22 kB)
  Using cached pyarrow-25.0.1-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8

In [3]:
!pip install importlib_resources

In [4]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt

# Load dataset with custom splits and metadata
(train_data, validation_data, test_data), info = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True
)

label_names = info.features['label'].names
print(f"Labels: {label_names}")


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cats_vs_dogs/incomplete.UOK8S4_4.0.1/cats_vs_dogs-train.tfrecord-[0-9][0-9…

Dataset cats_vs_dogs downloaded and prepared to /root/tensorflow_datasets/cats_vs_dogs/4.0.1. Subsequent calls will reuse this data.
Labels: ['cat', 'dog']


In [5]:
IMG_SIZE = 160  # All images will be resized to 160x160
BATCH_SIZE = 32

def format_example(image, label):
    # Resize image to a uniform shape
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    # Normalize pixel values from [0, 255] to [0, 1] or [-1, 1]
    image = image / 255.0
    return image, label

# Apply formatting, shuffle, and batch the datasets
train_batches = train_data.map(format_example).shuffle(1000).batch(BATCH_SIZE).prefetch(1)
validation_batches = validation_data.map(format_example).batch(BATCH_SIZE).prefetch(1)
test_batches = test_data.map(format_example).batch(BATCH_SIZE)


In [6]:
# Define a simple CNN architecture
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(64, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid') # Binary classification (0 = Cat, 1 = Dog)
])

# Compile the model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])




/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
# Train the model using the prepared batches
history = model.fit(train_batches, epochs=5, validation_data=validation_batches)

Epoch 1/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 47s 65ms/step - accuracy: 0.5615 - loss: 0.6991 - val_accuracy: 0.6023 - val_loss: 0.6716
Epoch 2/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 30s 49ms/step - accuracy: 0.6019 - loss: 0.6643 - val_accuracy: 0.6268 - val_loss: 0.6497
Epoch 3/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 29s 48ms/step - accuracy: 0.6436 - loss: 0.6286 - val_accuracy: 0.6595 - val_loss: 0.6225
Epoch 4/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 30s 50ms/step - accuracy: 0.6915 - loss: 0.5819 - val_accuracy: 0.6479 - val_loss: 0.6291
Epoch 5/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 30s 50ms/step - accuracy: 0.7415 - loss: 0.5142 - val_accuracy: 0.6612 - val_loss: 0.6746
